In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42

# True: clean and split, then for each fold run through the embedding and save the vectors.
# False: skip that work. The tuning loop loads those saved fold frames.
REBUILD = False

spark = SparkSession.builder \
                            .master("local[*]")\
                            .appName("DrugReviewClassification")\
                            .config("spark.driver.memory","4g")\
                            .getOrCreate()
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [ ]:
ANALYTICS_MODEL = "improved_baseline"
EMBEDDING_MODEL = "glove"
K = 5

PARA_RANGE = dict()

In [ ]:
from src.cleaning import data_cleaning
from src.embeddings import EmbeddingModel
from src.encoder import encoder
from src.k_flod import KFold
from src.split import split, weight
from src.text_features import text_feature_engineering, fit_phrases


embedding_train = getattr(EmbeddingModel, f"{EMBEDDING_MODEL}_train")
embedding_predict = getattr(EmbeddingModel, f"{EMBEDDING_MODEL}_predict")

if REBUILD:
    with data_oi.scope():
        df_raw = data_oi.load_raw_data()
        df_cleaned = data_cleaning(df_raw)
        df_cleaned.cache().count()
        df_engineered = text_feature_engineering(df_cleaned, True)
        df_engineered.cache().count()
        df_train, df_valid, df_test = split(df_engineered, SEED)
        fold_ids = iter(range(K))

        def prepare_fold(fit, holdout):
            fold = next(fold_ids)
            print(f"Fold {fold}")
            weighted = weight(fit)
            encoded_fit, encoded_holdout = encoder(weighted, holdout)
            encoded_fit.count()
            encoded_holdout.count()
            data_oi.unpersist(weighted)
            phrases = fit_phrases(encoded_fit)
            embedding_model = EmbeddingModel(
                encoded_fit, encoded_holdout, encoded_fit.limit(1), phrases
            )
            model_name = f"{EMBEDDING_MODEL}_para_{fold}"
            print(f"Fold {fold} Embedding Model `{EMBEDDING_MODEL}` Training")
            embedding_train(embedding_model, model_name)
            print(f"Fold {fold} Embedding Model `{EMBEDDING_MODEL}` Running")
            train_embedding, valid_embedding, test_embedding = embedding_predict(
                embedding_model, model_name
            )
            data_oi.unpersist(encoded_fit, encoded_holdout, test_embedding)
            for extra in (
                embedding_model._train_expanded,
                embedding_model._train_expanded_phrase,
            ):
                if extra is not None:
                    data_oi.unpersist(extra)
            del embedding_model, phrases
            return train_embedding, valid_embedding

        folds = KFold(f"{EMBEDDING_MODEL}_para", df_train, K, prepare_fold, SEED + 1)
else:
    folds = KFold.open(f"{EMBEDDING_MODEL}_para", K)

In [ ]:
from itertools import product

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F

from src.analytics import AnalyticsModel

analytics_function = getattr(AnalyticsModel, ANALYTICS_MODEL)
range_keys = list(PARA_RANGE)
candidates = [
    dict(zip(range_keys, values))
    for values in product(*(PARA_RANGE[key] for key in range_keys))
]
evaluator = RegressionEvaluator(labelCol="rating", predictionCol="rating_pred", metricName="mae")

In [ ]:
best_para = None
best_mae = None

for trial in candidates:
    folds.reset()
    fold_maes = []

    for i in range(K):
        with data_oi.scope():
            fit_embedding, holdout_embedding = folds.next_fold()
            model = AnalyticsModel(
                EMBEDDING_MODEL, fit_embedding, holdout_embedding, fit_embedding.limit(1)
            )
            _, holdout_pred, _ = analytics_function(model, trial)
            scored = holdout_pred.withColumn("rating_pred", F.col("prediction") + 1)
            fold_maes.append(evaluator.evaluate(scored))
            model.release_features()
            del fit_embedding, holdout_embedding, holdout_pred, scored, model

    mae = sum(fold_maes) / K
    print(trial, mae)
    if best_mae is None or mae < best_mae:
        best_mae = mae
        best_para = trial
        data_oi.save_analytics_para(EMBEDDING_MODEL, ANALYTICS_MODEL, best_para)
        print(f"saved best mae: {best_mae}")

para = data_oi.load_analytics_para(EMBEDDING_MODEL, ANALYTICS_MODEL)
print(f"best mae: {best_mae}")
print(para)